# Settimana 2 — Dall'effetto alla causa: alberi di probabilità e Teorema di Bayes

Questo è il notebook della Settimana 2 del corso di Probabilità e Statistica. Contiene gli script Python della [dispensa della settimana](https://qwertyjacob.github.io/PSI-VA-2627/settimana_02/), nello stesso ordine in cui compaiono: ogni script è una cella di codice, preceduta dal titolo della sezione in cui si trova. Le spiegazioni restano nella dispensa.

Per eseguire una cella, premi il pulsante ▶ alla sua sinistra. Gli script sono indipendenti l'uno dall'altro, quindi puoi eseguirli in qualsiasi ordine e modificarli liberamente. Colab non salva le modifiche nel notebook originale: se vuoi conservarle, scegli *File → Salva una copia in Drive*.

## Script 1 — Un calcolatore ad albero per qualunque partizione

[Leggi la spiegazione nella dispensa](https://qwertyjacob.github.io/PSI-VA-2627/settimana_02/#lab-albero)

In [ ]:
# Script 1 — Calcolatore ad albero: partizione {H_1,...,H_n}, evidenza E
def albero(a_priori, verosimiglianza):
    """a_priori: {H_i: P(H_i)}; verosimiglianza: {H_i: P(E | H_i)}.
    Restituisce foglie P(H_i ∩ E), P(E) (probabilità totale) e probabilità a posteriori P(H_i | E)."""
    assert abs(sum(a_priori.values()) - 1) < 1e-12, "le H_i devono formare una partizione"
    foglie = {h: a_priori[h] * verosimiglianza[h] for h in a_priori}    # regola del cammino
    p_E = sum(foglie.values())                                          # probabilità totale
    a_posteriori = {h: foglie[h] / p_E for h in a_priori}               # Bayes
    return foglie, p_E, a_posteriori

# (a) Test medico: prevalenza 1%, sensibilità 0.95, specificità 0.90
foglie, p_pos, post = albero({"H": 0.01, "H^c": 0.99}, {"H": 0.95, "H^c": 0.10})
print(f"Test medico: P(E) = {p_pos:.4f}, P(H|E) = {post['H']:.4f}")
assert abs(p_pos - 0.1085) < 1e-12 and round(post["H"], 3) == 0.088

# (b) Partizione con n = 4 ipotesi: moduli software (Baron 2.35), E = crash
a_priori = {"solo I": 0.12, "solo II": 0.32, "entrambi": 0.08, "nessuno": 0.48}
lik      = {"solo I": 0.5,  "solo II": 0.8,  "entrambi": 0.9,  "nessuno": 0.0}
_, p_C, post = albero(a_priori, lik)
print(f"Moduli: P(C) = {p_C:.3f}")
for h, v in post.items():
    print(f"  P({h} | C) = {v:.4f}")
assert abs(p_C - 0.388) < 1e-12 and round(post["entrambi"], 4) == 0.1856
assert abs(sum(post.values()) - 1) < 1e-12        # le probabilità a posteriori sommano a 1

# (c) Controllo qualità a tre livelli, letto all'indietro
p_T = {(True, True): 0.95, (True, False): 0.60, (False, True): 0.60, (False, False): 0.30}
foglie3 = {}
for m, pm in [(True, 0.9), (False, 0.1)]:
    pd = 0.8 if m else 0.4
    for d, pdd in [(True, pd), (False, 1 - pd)]:
        for t, pt in [(True, p_T[(m, d)]), (False, 1 - p_T[(m, d)])]:
            foglie3[(m, d, t)] = pm * pdd * pt                        # prodotto lungo il cammino
assert abs(sum(foglie3.values()) - 1) < 1e-12
p_Tc = sum(v for (m, d, t), v in foglie3.items() if not t)
dove = {(m, d): foglie3[(m, d, False)] / p_Tc for (m, d) in p_T}
print(f"Qualità: P(T) = {1 - p_Tc:.3f}, P(T^c) = {p_Tc:.3f}")
for (m, d), v in dove.items():
    print(f"  P({'M' if m else 'M^c'}∩{'D' if d else 'D^c'} | T^c) = {v:.4f}")
assert abs(p_Tc - 0.166) < 1e-12 and round(dove[(True, False)], 4) == 0.4337
print("Script 1: tutti gli assert superati.")

## Script 2 — Quanto conta la prevalenza

[Leggi la spiegazione nella dispensa](https://qwertyjacob.github.io/PSI-VA-2627/settimana_02/#lab-prevalenza)

In [ ]:
# Script 2 — posteriore_bayes e variazione della prevalenza
def posteriore_bayes(prevalenza, sensibilita, specificita):
    """P(H | E) dati P(H), sensibilità P(E | H) e specificità P(E^c | H^c)."""
    falsi_positivi = 1 - specificita
    numeratore = sensibilita * prevalenza
    return numeratore / (numeratore + falsi_positivi * (1 - prevalenza))

casi = [("Test medico (1%, 0.95, 0.90)", 0.01, 0.95, 0.90, 0.088, 3),
        ("IDS del Prologo (1%, 99%, 99%)", 0.01, 0.99, 0.99, 0.5, 6),
        ("Baron 2.34 (4%, 0.95, 0.99)", 0.04, 0.95, 0.99, 0.7983, 4)]
for nome, prev, sens, spec, atteso, cifre in casi:
    p = posteriore_bayes(prev, sens, spec)
    print(f"{nome:34s} P(H|E) = {p:.4f}")
    assert round(p, cifre) == atteso

print("IDS con sensibilità = specificità = 0.99, al variare della prevalenza:")
for prev in [0.5, 0.1, 0.01, 0.001, 0.0001]:
    print(f"  prevalenza {prev:<7} -> P(attacco|allarme) = {posteriore_bayes(prev, 0.99, 0.99):.4f}")
assert posteriore_bayes(0.0001, 0.99, 0.99) < 0.01          # meno di un allarme su cento
print("Script 2: tutti gli assert superati.")

## Script 3 — Simulare 200.000 connessioni

[Leggi la spiegazione nella dispensa](https://qwertyjacob.github.io/PSI-VA-2627/settimana_02/#lab-montecarlo)

In [ ]:
# Script 3 — Monte Carlo IDS: la frequenza converge alla probabilità a posteriori di Bayes
import random
random.seed(2627)
N, prev, sens, fpr = 200_000, 0.01, 0.99, 0.01
allarmi = veri = 0
for _ in range(N):
    attacco = random.random() < prev
    allarme = random.random() < (sens if attacco else fpr)
    if allarme:
        allarmi += 1
        veri += attacco
freq = veri / allarmi
bayes = sens * prev / (sens * prev + fpr * (1 - prev))
print(f"Connessioni simulate: {N}")
print(f"Allarmi: {allarmi}, di cui veri: {veri}")
print(f"Frequenza empirica P(attacco|allarme) = {freq:.4f}  (Bayes: {bayes:.4f})")
assert abs(bayes - 0.5) < 1e-12
assert abs(freq - bayes) < 0.04          # deviazione standard attesa ≈ 0.008
print("Script 3: tutti gli assert superati.")

## Script 4 — Il secondo allarme

[Leggi la spiegazione nella dispensa](https://qwertyjacob.github.io/PSI-VA-2627/settimana_02/#lab-sequenziale)

In [ ]:
# Script 4 — Aggiornamento sequenziale vs in blocco (rilevatori condizionatamente indipendenti)
import random
def aggiorna(a_priori, p_E_H, p_E_Hc):
    return p_E_H * a_priori / (p_E_H * a_priori + p_E_Hc * (1 - a_priori))

a_priori = 0.01
d1 = (0.99, 0.01)     # (sensibilità, tasso di falsi positivi) del rilevatore di rete
d2 = (0.95, 0.05)     # rilevatore sul server, condizionatamente indipendente dato H
post1 = aggiorna(a_priori, *d1)
post12 = aggiorna(post1, *d2)                                   # la probabilità a posteriori diventa la nuova probabilità a priori
blocco = aggiorna(a_priori, d1[0] * d2[0], d1[1] * d2[1])       # E1 ∩ E2 in un colpo solo
print(f"Dopo E1: {post1:.4f} | dopo E1 poi E2: {post12:.4f} | in blocco: {blocco:.4f}")
assert round(post1, 6) == 0.5 and abs(post12 - blocco) < 1e-12
assert round(blocco, 6) == 0.95

# Verifica Monte Carlo: si generano E1 ed E2 indipendenti *dato* H
random.seed(7)
n12 = h12 = 0
for _ in range(400_000):
    h = random.random() < a_priori
    e1 = random.random() < (d1[0] if h else d1[1])
    e2 = random.random() < (d2[0] if h else d2[1])
    if e1 and e2:
        n12 += 1; h12 += h
print(f"Monte Carlo P(H|E1∩E2) = {h12/n12:.4f} su {n12} doppi allarmi")
assert abs(h12 / n12 - blocco) < 0.02
print("Script 4: tutti gli assert superati.")

## Script 5 — Simulare i moduli software

[Leggi la spiegazione nella dispensa](https://qwertyjacob.github.io/PSI-VA-2627/settimana_02/#lab-moduli)

In [ ]:
# Script 5 — Monte Carlo dei moduli software (Baron, Esempio 2.35)
import random
random.seed(235)
N = 400_000
crash = entrambi = 0
for _ in range(N):
    a = random.random() < 0.2          # errore nel modulo I
    b = random.random() < 0.4          # errore nel modulo II (indipendente)
    p_c = 0.9 if (a and b) else 0.5 if a else 0.8 if b else 0.0
    if random.random() < p_c:
        crash += 1
        entrambi += (a and b)
esatto = 0.9 * 0.08 / (0.5 * 0.12 + 0.8 * 0.32 + 0.9 * 0.08)
print(f"P(C) empirica = {crash/N:.4f} (esatta 0.388)")
print(f"P(A∩B|C) empirica = {entrambi/crash:.4f} (esatta {esatto:.4f})")
assert round(esatto, 4) == 0.1856
assert abs(crash / N - 0.388) < 0.005 and abs(entrambi / crash - esatto) < 0.01
print("Script 5: tutti gli assert superati.")

## Script 6 — Il tavolo di Bayes

[Leggi la spiegazione nella dispensa](https://qwertyjacob.github.io/PSI-VA-2627/settimana_02/#lab-tavolo)

In [ ]:
# Script 6 — Il tavolo di Bayes discretizzato
import random
def tavolo_a_posteriori(p, q, strisce):
    """Probabilità a posteriori delle strisce di W dopo p eventi M e q eventi M^c,
    partendo da probabilità a priori uniformi."""
    centri = [(j + 0.5) / strisce for j in range(strisce)]
    pesi = [x**p * (1 - x)**q for x in centri]       # probabilità a priori uniforme × verosimiglianza
    Z = sum(pesi)                                    # normalizzazione (probabilità totale)
    return centri, [w / Z for w in pesi]

random.seed(1763)
w = random.random()                                  # posizione (nascosta) della palla W
lanci = [random.random() < w for _ in range(20)]     # evento M: O cade dalla parte "di qua"
print(f"Posizione nascosta di W: {w:.3f}")
for k in (0, 1, 5, 20):
    p = sum(lanci[:k]); q = k - p
    centri, post = tavolo_a_posteriori(p, q, 10)
    assert abs(sum(post) - 1) < 1e-12
    barre = " ".join(f"{v:.2f}" for v in post)
    print(f"k={k:2d} (p={p:2d}, q={q:2d}): {barre}")

# Con strisce sottilissime si ritrovano i risultati di Price e Laplace
centri, post = tavolo_a_posteriori(1, 0, 10_000)
p_oltre_meta = sum(v for x, v in zip(centri, post) if x > 0.5)
print(f"Dopo un'alba: P(x > 1/2) = {p_oltre_meta:.4f}  -> {p_oltre_meta/(1-p_oltre_meta):.2f} contro 1")
assert abs(p_oltre_meta - 0.75) < 1e-3
centri, post = tavolo_a_posteriori(7, 3, 10_000)
media = sum(x * v for x, v in zip(centri, post))
print(f"p=7, q=3: media a posteriori = {media:.4f}  (regola di successione (p+1)/(n+2) = {8/12:.4f})")
assert abs(media - 8 / 12) < 1e-4
n = 1_826_213                                         # giorni in 5000 anni (Laplace)
print(f"Sole di Laplace: (n+1) contro 1 = {n + 1} contro 1")
print("Script 6: tutti gli assert superati.")